# walkman: basic dashboard

Launched by `walkman start <instrument>`. Fill in the inputs and click **Run**.

In [ ]:
%matplotlib widget
import os
import threading
from html import escape
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

from walkman.backend import NoiseAnalysis
from walkman.instruments import CONFIG_ENV_VAR, INSTRUMENT_ENV_VAR, OUTPUT_ENV_VAR, load_instrument
from walkman.plots import (HistogramGrid, NoiseTrendPlot, PSDPlot, noise_label_html, noise_value_html,
                           series_color)

missing = [var for var in (INSTRUMENT_ENV_VAR, CONFIG_ENV_VAR) if var not in os.environ]
if missing:
    raise KeyError(f"Environment variables {missing} are not set. Launch this notebook with `walkman start <instrument>`.")

analysis = NoiseAnalysis(load_instrument(os.environ[INSTRUMENT_ENV_VAR]), os.environ[CONFIG_ENV_VAR])

In [ ]:
wide = {"description_width": "initial"}
input_paths = widgets.Textarea(description="Data folder(s)", placeholder="absolute/path/to/data, one path or glob per line",
                               style=wide, layout=widgets.Layout(width="60%"))
output_dir = widgets.Text(value=os.environ.get(OUTPUT_ENV_VAR, ""), description="Output folder",
                          placeholder="absolute/path/to/results", style=wide, layout=widgets.Layout(width="60%"))
watch_mode = widgets.Checkbox(value=False, description="Watch mode", indent=False)
poll_interval = widgets.BoundedFloatText(value=5.0, min=0.0, max=3600.0, step=0.5, description="Poll interval (s)",
                                         style=wide, disabled=True)
settle_time = widgets.BoundedFloatText(value=2.0, min=0.0, max=3600.0, step=0.5, description="Settle time (s)",
                                       style=wide, disabled=True)
# noise analysis inputs
region_boxes = {region: widgets.Checkbox(value=region == "overscan", description=region, indent=False)
                for region in ("full", "image", "overscan")}
trim_start = widgets.BoundedIntText(value=0, min=0, max=100000, description="Trim start", style=wide)
trim_end = widgets.BoundedIntText(value=0, min=0, max=100000, description="Trim end", style=wide)
target_shape = widgets.BoundedIntText(value=0, min=0, max=100000, description="Target shape (0 = auto)", style=wide)
window_func = widgets.Text(value="hamming", description="Window function", style=wide)
run_button = widgets.Button(description="Run", button_style="primary", icon="play")
stop_button = widgets.Button(description="Stop", icon="stop", disabled=True,
                             tooltip="Stop after the current image")
reset_button = widgets.Button(description="Reset seen files", icon="undo",
                              tooltip="Forget the files processed by earlier watch mode runs")
status = widgets.HTML()
stop_requested = threading.Event()

# poll interval and settle time only matter in watch mode
for field in (poll_interval, settle_time):
    widgets.dlink((watch_mode, "value"), (field, "disabled"), transform=lambda watch: not watch)

# dashboard widgets (noise table, combined PSD, noise trend, histogram grid), refreshed after every image.
# result_rows maps (det_id, output, region) to its color, table widgets and latest results, in order of first appearance.
result_rows = {}
table_header = [widgets.HTML(f"<b style='font-size:14px;color:#52514e'>{name}</b>") for name in ("", "Noise", "PSD")]
noise_table = widgets.GridBox(table_header, layout=widgets.Layout(
    width="50%", grid_template_columns="1fr auto auto", grid_gap="4px 16px", align_items="center",
    max_height="400px", overflow_y="auto"))
psd_plot = PSDPlot()
trend_plot = NoiseTrendPlot(xlabel=f"Time (from header KEY: {analysis.task.meta['timestamp_key']}, displayed in MJD)")
hist_grid = HistogramGrid()
dashboard = widgets.VBox([widgets.HBox([noise_table, psd_plot.widget]), trend_plot.widget, hist_grid.widget],
                         layout=widgets.Layout(display="none"))

def _add_table_row(key, color):
    noise_w = widgets.HTML()
    show = widgets.Checkbox(value=True, indent=False, layout=widgets.Layout(width="auto"))

    def _toggle(change):
        psd_plot.set_visible(key, change["new"])
        trend_plot.set_visible(key, change["new"])

    show.observe(_toggle, names="value")
    noise_table.children = (*noise_table.children, widgets.HTML(noise_label_html(*key, color)), noise_w, show)
    return noise_w

def _update_rows(noise_df):
    for rec in noise_df.itertuples(index=False):
        key = (rec.det_id, rec.output, rec.region)
        if key not in result_rows:
            # colors follow the row identity, so they stay fixed across refreshes
            color = series_color(len(result_rows))
            result_rows[key] = {"color": color, "noise_w": _add_table_row(key, color)}
        row = result_rows[key]
        row.update(psd=rec.psd, histogram=rec.histogram)
        row["noise_w"].value = noise_value_html(rec.noise, row["color"])
    psd_plot.update({key: (row["color"], *row["psd"]) for key, row in result_rows.items()})
    # the trend uses the results accumulated by the backend, so it shows every image processed so far
    trend_plot.update({key: (result_rows[key]["color"], group["time"].astype(float), group["noise"])
                       for key, group in analysis.result.data.groupby(["det_id", "output", "region"], sort=False)
                       if key in result_rows})
    hist_grid.update({key: (" · ".join(map(str, key)), row["color"], *row["histogram"])
                      for key, row in result_rows.items()})
    dashboard.layout.display = None

def _clear_rows():
    result_rows.clear()
    noise_table.children = table_header
    psd_plot.clear()
    trend_plot.clear()
    hist_grid.clear()
    dashboard.layout.display = "none"

def _set_running(running):
    run_button.disabled = running
    reset_button.disabled = running
    stop_button.disabled = not running

def _run(paths, out_dir, params):
    # Runs in a background thread so the kernel stays responsive
    _clear_rows()
    n_images = 0
    try:
        for noiseres in analysis.run(paths, out_dir, **params):
            n_images += 1
            _update_rows(noiseres.data)
            status.value = f"Processed {n_images} image(s), saved to {escape(str(analysis.run_dir))}"
        done = "Stopped" if stop_requested.is_set() else "Analysis complete"
        status.value = f"{done}: {n_images} image(s) saved to {escape(str(analysis.run_dir))}"
    except Exception as e:
        # Exceptions in threads are otherwise lost, so surface them in the UI
        status.value = f"<b style='color:red'>Error:</b> {escape(type(e).__name__)}: {escape(str(e))}"
    finally:
        _set_running(False)

def on_run(_):
    paths = [str(Path(line.strip()).expanduser()) for line in input_paths.value.splitlines() if line.strip()]
    out_dir = Path(output_dir.value.strip()).expanduser()
    regions = [region for region, box in region_boxes.items() if box.value]
    error = ("enter at least one data folder." if not paths else
             "data paths must be absolute." if not all(Path(path).is_absolute() for path in paths) else
             "output folder must be absolute." if not out_dir.is_absolute() else
             "select at least one region." if not regions else None)
    if error:
        status.value = f"<b style='color:red'>Error:</b> {error}"
        return
    params = dict(regions=regions, trim_start=trim_start.value, trim_end=trim_end.value,
                  target_shape=target_shape.value or None, window_func=window_func.value or "hamming",
                  watch=watch_mode.value, poll_interval=poll_interval.value, settle_time=settle_time.value)
    stop_requested.clear()
    _set_running(True)
    status.value = f"Watching every {poll_interval.value} s..." if watch_mode.value else "Running..."
    threading.Thread(target=_run, args=(paths[0] if len(paths) == 1 else paths, str(out_dir), params),
                     daemon=True).start()

def on_stop(_):
    stop_requested.set()
    analysis.stop()
    stop_button.disabled = True
    status.value += "<br>Stopping after the current image..."

def on_reset(_):
    analysis.reset()
    status.value = "Seen files cleared, the next watch mode run processes all files again."

run_button.on_click(on_run)
stop_button.on_click(on_stop)
reset_button.on_click(on_reset)
display(widgets.VBox([
    input_paths,
    output_dir,
    widgets.HBox([watch_mode, poll_interval, settle_time]),
    widgets.HBox([widgets.Label("Regions:"), *region_boxes.values()]),
    widgets.HBox([trim_start, trim_end]),
    widgets.HBox([target_shape, window_func]),
    widgets.HBox([run_button, stop_button, reset_button]),
    status,
    dashboard,
]))